# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [1]:
%%sh
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint:
hint: 	git config --global init.defaultBranch <name>
hint:
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint:
hint: 	git branch -m <name>
hint:
hint: Disable this message with "git config set advice.defaultBranchName false"


Инициализирован пустой репозиторий Git в /Users/v.gorlishchev/Documents/Магистратура/Семестр 2/HW7_CICD/.git/


In [2]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


### Проверяем работоспособность пайплайна

In [3]:
!python ml_pipeline.py

/Users/v.gorlishchev/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
Точность аccuracy: 1.00


In [4]:
%%writefile .gitlab-ci.yml
build:
  image: python:3.11-slim
  script:
    - pip install -r requirements.txt
    - python ml_pipeline.py
  after_script:
    - echo "Make pipeline reproducible"
    - pip freeze > requirements.lock.txt


Overwriting .gitlab-ci.yml


In [5]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log

[master (корневой коммит) 2ff9ced] build(ml_pipeline.py) добавлен пайплайн GitLab
 2 files changed, 26 insertions(+)
 create mode 100644 .gitlab-ci.yml
 create mode 100644 ml_pipeline.py
commit 2ff9cedbdb93314a6c83af1a88ba26b7acc6a427 (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Mon May 11 20:41:06 2026 +0300

    build(ml_pipeline.py) добавлен пайплайн GitLab


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

Пайплайн GitLab: https://gitlab.com/gorlishev.v/mlops-hw7-cicd/-/pipelines

Файлы: https://gitlab.com/gorlishev.v/mlops-hw7-cicd

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

Выбрал Blue-Green. Это когда рядом работают две версии модели — старая и новая. Балансер сначала шлёт всех пользователей в старую, а новая в это время прогревается. Когда новая готова — переключаем на неё одной строчкой в конфиге. Если что-то пойдёт не так, возвращаемся на старую за секунду.

Сравнивал с Canary. Canary сразу пускает часть людей в новую версию (например 10%). У нас в коде нет обработки ошибок, поэтому даже маленькая доля сломанного трафика — это уже неприятно для пользователей.

Из минусов Blue-Green: на время деплоя памяти на сервере нужно в два раза больше, потому что крутятся обе версии.

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [6]:
%%writefile docker-compose-blue.yaml
services:
  ml-blue:
    build: .
    image: ml:v1.0.0
    environment: { MODEL_VERSION: v1.0.0 }
    ports: ["8001:8000"]


Writing docker-compose-blue.yaml


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

Сравниваю две модели:
- A — старая, v1.0.0
- B — новая, v1.1.0

Делим пользователей пополам. Главная метрика — accuracy (процент правильных ответов). Чтобы её померить, на каждом сотом запросе проверяем ответ вручную.

Чтобы заметить разницу хотя бы в 1 процентный пункт по качеству, нужно собрать примерно 4700 запросов на каждую группу. При обычной нагрузке это меньше часа.

Если у новой модели качество заметно ниже или ответы стали медленнее — откатываемся обратно на старую.

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [7]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [8]:
!python ml_pipeline.py

/Users/v.gorlishchev/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [9]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
      with: { python-version: '3.11' }
    - run: pip install -r requirements.txt
    - run: python ml_pipeline.py
    - name: Make pipeline reproducible
      run: pip freeze > requirements.lock.txt


Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [10]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [11]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master 9327bef] build(ml_pipeline.py) добавлен пайплайн GitHub Actions
 1 file changed, 14 insertions(+)
 create mode 100644 .github/workflows/ci.yml
commit 9327bef44f8e4180599a27db194aa96c33548444 (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Mon May 11 20:41:08 2026 +0300

    build(ml_pipeline.py) добавлен пайплайн GitHub Actions

commit 2ff9cedbdb93314a6c83af1a88ba26b7acc6a427
Author: Your Name <you@example.com>
Date:   Mon May 11 20:41:06 2026 +0300

    build(ml_pipeline.py) добавлен пайплайн GitLab


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

Пайплайн GitHub Actions: https://github.com/gorlishevv-jpg/mipt-homeworks/actions

Файлы HW7: https://github.com/gorlishevv-jpg/mipt-homeworks/tree/main/mlops-hw7

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.




GitHub Actions делать просто — синтаксис понятный,заполнил шаблон и пайплайн запустился. 

Стратегию выбрал Blue-Green: она самая понятная и откат делается мгновенно.

Про воспроизводимость — недостаточно сохранить только код, нужны ещё точные версии библиотек и Python, иначе пайплайн сложно будет повторить